# Table 2 — pooled performance (SPATNIC)

**🟢 light (reads caches)** · source: `notebooks/metrics_pooled_persample_all.py`

⏱ B=1000 bootstrap over ~577k external cells → ~5-10 min.

## Configuration — edit the paths, then run

In [ ]:
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")
try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print("[note] scanpy/anndata not available:", e)

# ── EDIT THESE PATHS to match your environment ──
REPO_ROOT     = Path("/path/to/spatnic")          # this repository
BACKUP_ROOT   = Path("/path/to/backup")           # integrate_adata_filtered.h5ad, galaxy scores, Liver meta
DATA_ROOT     = Path("/path/to/data")             # GxD concat, lung annotated, c2l refs, spatnic_models, GxD_Xenium
BENCHMARK_DB  = Path("/path/to/benchmark_db")     # Xenium/VisiumHD/MERFISH/CosMx + adata_hvg_*
VISIUMHD_ROOT = Path("/path/to/VisiumHD")         # Visium HD ADC track
WEIGHTS_DIR   = Path.home() / ".spatnic" / "weights"

# ── Derived ──
NB     = REPO_ROOT / "notebooks"
COMP   = NB / "comparison_results"
VHD    = VISIUMHD_ROOT
MODELS = DATA_ROOT / "spatnic_models"
PAPER  = REPO_ROOT / "paper"
BASE_DIR  = VHD          # VisiumHD ADC notebook global
THRESHOLD = 0.9          # overridden to 0.5 by the Fig 5 shortcut setup cell
sys.path[:0] = [str(REPO_ROOT / "scripts"), str(NB)]
if NB.exists():
    os.chdir(NB)         # extracted cells were written for cwd = notebooks/

def _tbl(csv, n=None):
    p = Path(csv)
    if not p.exists():
        print("[missing]", p); return None
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
    display(df.head(n) if n else df); return df

def _run(script, show=None, n=None):
    import subprocess
    cmd = f"python notebooks/{script}"
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    print((r.stdout or "")[-3000:])
    if r.returncode: print("STDERR:\n", (r.stderr or "")[-2000:])
    if show: _tbl(COMP / show, n)


## Regenerate (runs the real metric program)

In [ ]:
_run("metrics_pooled_persample_all.py", show="eval_confmat/metrics_pooled.csv")

## Result (current cached values)

**Pooled (cell-level, 95% bootstrap CI)** (`metrics_pooled.csv`, 4 rows)

| config | n_cells | AUROC | AUPRC | MCC | F1 | Sensitivity | Specificity | BalancedAccuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| primary model on primary test (CRC Xeniu | 97804 | 0.991 [0.991–0.992] | 0.990 [0.990–0.991] | 0.920 [0.918–0.923] | 0.960 [0.959–0.961] | 0.961 [0.959–0.963] | 0.959 [0.957–0.961] | 0.960 [0.959–0.961] |
| primary model on primary external test ( | 673701 | 0.984 [0.984–0.984] | 0.997 [0.997–0.997] | 0.825 [0.824–0.827] | 0.969 [0.969–0.970] | 0.951 [0.950–0.951] | 0.941 [0.940–0.942] | 0.946 [0.945–0.947] |
| liver met model on liver met test (GxD,  | 273756 | 0.992 [0.992–0.992] | 0.996 [0.995–0.996] | 0.923 [0.921–0.925] | 0.977 [0.977–0.978] | 0.978 [0.977–0.978] | 0.945 [0.944–0.947] | 0.961 [0.961–0.962] |
| lung met model on lung met test (GxD, 9- | 55076 | 0.998 [0.997–0.998] | 0.999 [0.999–1.000] | 0.930 [0.926–0.934] | 0.987 [0.986–0.988] | 0.977 [0.975–0.978] | 0.988 [0.986–0.990] | 0.982 [0.981–0.984] |